# Snowflake — VKG Query-Agent Eval (live QA over the federated layer)

Step **2 of 2**. `17_snowflake_semantic_layer.ipynb` built the VKG layer over the
live Snowflake tables; this notebook **asks it natural-language questions** through
the deployed **ontology_query** agent (VKG path, `QueryRuntimeArn`) and scores the
answers.

Structurally it mirrors `14_cwd_benchmark_eval.ipynb`, with two deliberate
differences driven by the fact that the data is in **Snowflake**, not the lake:

| | CWD (nb14) | Here |
|:--|:--|:--|
| Questions + gold | `data/eval/benchmarks/cwd/cwd.jsonl` (shipped with the benchmark) | **authored in §3 of this notebook** — there is no Snowflake benchmark file. Each question ships hand-written gold SQL. |
| Exec metrics (EX / R-VES / Soft-F1) | three `codeBased` TRACE evaluators → the `BenchmarkEvalHandler` Lambda | **computed locally in §7** (see below) |
| Expected answer for the judges | the packed gold **SQL** JSON, so `FinalAnswerFaithfulness` is weak/uninformative | a **prose answer rendered from the live gold result**, so `FinalAnswerFaithfulness` is meaningful here |

### Why exec metrics are computed locally rather than by the codeBased Lambda
The `BenchmarkEvalHandler` Lambda executes gold SQL on Athena itself, so it needs
whatever the query needs. Athena runs a **federated** query by invoking the
connector Lambda **as the query's own principal**, which therefore needs
`lambda:InvokeFunction` on the connector function. The handler's role was built for
the S3 Tables benchmarks: it has the full Athena + Glue + `s3tables` +
`lakeformation:GetDataAccess` set but **no `lambda:` action at all**, so every gold
query against `snowflake_*` fails at submit time. Registering the codeBased
evaluators anyway would yield `null`/0 scores indistinguishable from agent failures.
So §3 executes each gold query **from this notebook** (which does have access — the
same path §3 of nb17 uses) and §8 compares the agent's returned rows against those
gold rows.

> **Follow-up (not done here):** add `lambda:InvokeFunction` on the connector
> function to the three `BenchmarkEvalHandler*` roles, and nb14's exact three
> `codeBased` evaluators can be re-enabled for Snowflake — which would also restore
> R-VES and Soft-F1, which this notebook does not compute.

### What is scored
| Signal | Kind | Level | Where |
|:--|:--|:--|:--|
| `GoalSuccess` | LLM-as-Judge | SESSION | `eval_judges` (VKG family), server-side |
| `FinalAnswerFaithfulness` | LLM-as-Judge | SESSION | same |
| `SqlGrounded` | LLM-as-Judge | SESSION | same |
| `ExecutionAccuracy` (local) | result-set comparison | per question | §7 of this notebook |

### Prerequisites
- **`17_snowflake_semantic_layer.ipynb` completed**, leaving a `completed` layer at
  `vkg-snowflake_<schema>` (default `vkg-snowflake_fhir_r4`). §2 verifies this in
  DynamoDB and refuses to run otherwise.
- The connector still healthy (`npm run set:snowflake-casing` survives redeploys —
  a deploy resets the connector Lambda's environment).


In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv, set_key
from IPython.display import Markdown, display  # noqa: F401

# Load .env if present, then set defaults
ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

✓ AWS Credentials Verified:
  Profile: huthmac+demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [4]:
# ── CloudFormation output resolver ──
# For any env var still set to 'XXX' (no .env or .env missing the key), resolve from
# live CloudFormation stack outputs. Stacks may be deployed with a 'dev' suffix.
def _get_stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}[-dev]-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            resp = cfn.describe_stacks(StackName=name)
            return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

_CFN_MAP = {
    'ATHENA_WORKGROUP':        ('athena',   'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE': ('dynamodb', 'MetadataTableName'),
}
_cfn_cache, _resolved_from_cfn = {}, []
for env_var, (stack, output_key) in _CFN_MAP.items():
    if os.environ.get(env_var, 'XXX') != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)
if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

for key in ('ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE'):
    print(f"  {key} = {os.environ.get(key, '(not set)')}")

  ATHENA_WORKGROUP = semantic-layer-dev-workgroup
  ONTOLOGY_METADATA_TABLE = semantic-layer-dev-metadata


In [5]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Bind to the VKG layer that notebook 17 built

The layer id is **derived, not looked up**: the same two env-overridable constants
nb17 used rebuild it exactly (`vkg-snowflake_<schema lower-cased>`), so the two
notebooks cannot drift apart. It is then verified in DynamoDB — a missing or
non-`completed` layer stops the notebook, because querying an unbuilt layer returns
plausible-looking empty answers rather than an error.

In [6]:
# ── Derive the layer id exactly as notebook 17 did, then verify it is built ──
SNOWFLAKE_CATALOG = os.environ.get('SNOWFLAKE_ATHENA_CATALOG', 'snowflake_fhir_db')
SNOWFLAKE_SCHEMA = os.environ.get('SNOWFLAKE_ATHENA_SCHEMA', 'FHIR_R4')
LAYER_ID = f'vkg-snowflake_{SNOWFLAKE_SCHEMA.lower()}'   # SAME derivation as nb17

METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
_meta_table = session.resource('dynamodb', region_name=region).Table(METADATA_TABLE)

_item = _meta_table.get_item(Key={'id': LAYER_ID, 'version': 'v1'}).get('Item')
if not _item:
    raise RuntimeError(
        f"No semantic layer '{LAYER_ID}' in {METADATA_TABLE}. Run "
        f"17_snowflake_semantic_layer.ipynb first — it builds this exact id from "
        f"SNOWFLAKE_ATHENA_CATALOG/SNOWFLAKE_ATHENA_SCHEMA "
        f"({SNOWFLAKE_CATALOG}/{SNOWFLAKE_SCHEMA})."
    )
if _item.get('status') != 'completed':
    raise RuntimeError(
        f"Layer '{LAYER_ID}' exists but status={_item.get('status')!r} "
        f"(phase={_item.get('phase')!r}). Querying an unbuilt layer yields empty or "
        f"invented answers instead of an error, so refusing to evaluate. Re-run "
        f"notebook 17 and check the DynamoDB item's `error` field."
    )

_layer_tables = sorted({d['tableName'] for d in (_item.get('dataSources') or [])})
print(f"✓ Layer '{LAYER_ID}' is completed")
print(f"  name      : {_item.get('name')}")
print(f"  built     : {_item.get('completedAt', _item.get('updatedAt'))}")
print(f"  catalog   : {SNOWFLAKE_CATALOG}   schema: {SNOWFLAKE_SCHEMA}")
print(f"  tables    : {len(_layer_tables)} → {_layer_tables}")

✓ Layer 'vkg-snowflake_fhir_r4' is completed
  name      : snowflake-fhir_r4-ontology-20260813-030435
  built     : 2026-08-13T03:40:34.879176+00:00
  catalog   : snowflake_fhir_db   schema: FHIR_R4
  tables    : 6 → ['CONDITIONS', 'ENCOUNTERS', 'FHIR_RESOURCES', 'MEDICATIONS', 'OBSERVATIONS', 'PATIENTS']


## 3. QA question set + live gold execution

There is no Snowflake benchmark file, so the questions and their gold SQL are
authored here. Two deliberate properties:

- **Aggregate-shaped.** Every gold query returns at most 5 rows, so the §7
  comparison is never confounded by row-limit truncation in the agent's response
  payload.
- **Gold values are recomputed live**, not hard-coded. The tables can be reloaded,
  so a pinned expected number would silently rot into a false failure. The prose
  expected answer the LLM judges see is rendered from whatever the gold query
  returns *now*.

Seven questions are single-table aggregations; `sf-q8` is a **join** and is the
hardest case by design — the foreign key is `ENCOUNTERS.PATIENT_REF =
'urn:uuid:' || PATIENTS.ID`, i.e. a prefixed reference rather than a bare key, so
it tests whether the generated OBDA mapping expresses the relationship at all.

In [7]:
# ── The QA question set (hand-authored; gold SQL verified against Athena) ────
# `difficulty` is used only to break the results down; `tables` documents what each
# question touches. All gold SQL runs in QueryExecutionContext
# {Catalog: SNOWFLAKE_CATALOG, Database: SNOWFLAKE_SCHEMA}, so table names are bare.
QUESTIONS = [
    {
        'id': 'sf-q1', 'difficulty': 'simple', 'tables': ['PATIENTS'],
        'question': 'How many patients are in the dataset?',
        'gold_sql': 'SELECT count(*) AS patient_count FROM PATIENTS',
    },
    {
        'id': 'sf-q2', 'difficulty': 'simple', 'tables': ['PATIENTS'],
        'question': 'How many patients are there of each gender?',
        'gold_sql': ('SELECT GENDER, count(*) AS patient_count FROM PATIENTS '
                     'GROUP BY GENDER ORDER BY patient_count DESC'),
    },
    {
        'id': 'sf-q3', 'difficulty': 'simple', 'tables': ['CONDITIONS'],
        'question': 'How many conditions have a clinical status of active?',
        'gold_sql': ("SELECT count(*) AS active_conditions FROM CONDITIONS "
                     "WHERE CLINICAL_STATUS = 'active'"),
    },
    {
        'id': 'sf-q4', 'difficulty': 'moderate', 'tables': ['CONDITIONS'],
        'question': ('What are the five most common condition names, and how many '
                     'times does each occur?'),
        'gold_sql': ('SELECT CONDITION_NAME, count(*) AS condition_count '
                     'FROM CONDITIONS GROUP BY CONDITION_NAME '
                     'ORDER BY condition_count DESC, CONDITION_NAME ASC LIMIT 5'),
    },
    {
        'id': 'sf-q5', 'difficulty': 'moderate', 'tables': ['PATIENTS'],
        'question': 'Which three cities have the most patients?',
        'gold_sql': ('SELECT CITY, count(*) AS patient_count FROM PATIENTS '
                     'GROUP BY CITY ORDER BY patient_count DESC, CITY ASC LIMIT 3'),
    },
    {
        'id': 'sf-q6', 'difficulty': 'moderate', 'tables': ['ENCOUNTERS'],
        'question': 'How many encounters are there of each encounter class?',
        'gold_sql': ('SELECT ENCOUNTER_CLASS, count(*) AS encounter_count '
                     'FROM ENCOUNTERS GROUP BY ENCOUNTER_CLASS '
                     'ORDER BY encounter_count DESC, ENCOUNTER_CLASS ASC'),
    },
    {
        'id': 'sf-q7', 'difficulty': 'moderate', 'tables': ['MEDICATIONS'],
        'question': 'How many medication prescriptions are there per status?',
        'gold_sql': ('SELECT STATUS, count(*) AS prescription_count '
                     'FROM MEDICATIONS GROUP BY STATUS '
                     'ORDER BY prescription_count DESC, STATUS ASC'),
    },
    {
        # Hardest by design: the FK is a PREFIXED reference
        # (ENCOUNTERS.PATIENT_REF = 'urn:uuid:' || PATIENTS.ID), not a bare key.
        'id': 'sf-q8', 'difficulty': 'challenging',
        'tables': ['ENCOUNTERS', 'PATIENTS'],
        'question': 'How many encounters belong to female patients?',
        'gold_sql': ("SELECT count(*) AS encounter_count FROM ENCOUNTERS e "
                     "JOIN PATIENTS p ON e.PATIENT_REF = 'urn:uuid:' || p.ID "
                     "WHERE p.GENDER = 'female'"),
    },
]

MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    QUESTIONS = QUESTIONS[:MAX_ROWS]
    print(f"⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(QUESTIONS)}-question slice (0 = all)")

_missing_tables = sorted({t for q in QUESTIONS for t in q['tables']} - set(_layer_tables))
if _missing_tables:
    raise RuntimeError(
        f"Questions reference table(s) that are not in layer '{LAYER_ID}': "
        f"{_missing_tables}. The layer was built over {_layer_tables}. Either "
        f"rebuild with MAX_TABLES=0 in notebook 17, or drop those questions — the "
        f"agent cannot answer about a table the ontology does not model."
    )
print(f"✓ {len(QUESTIONS)} QA question(s); all referenced tables are in the layer")
display(pd.DataFrame([{'id': q['id'], 'difficulty': q['difficulty'],
                       'question': q['question']} for q in QUESTIONS]))

✓ 8 QA question(s); all referenced tables are in the layer


,id,difficulty,question
0,sf-q1,simple,How many patients are in the dataset?
1,sf-q2,simple,How many patients are there of each gender?
2,sf-q3,simple,How many conditions have a clinical status of active?
3,sf-q4,moderate,"What are the five most common condition names, and how many times does each occur?"
4,sf-q5,moderate,Which three cities have the most patients?
5,sf-q6,moderate,How many encounters are there of each encounter class?
6,sf-q7,moderate,How many medication prescriptions are there per status?
7,sf-q8,challenging,How many encounters belong to female patients?


In [8]:
# ── Execute every gold query on Athena to get the live ground truth ─────────
athena = session.client('athena', region_name=region)
ATHENA_WORKGROUP = os.environ['ATHENA_WORKGROUP']
# Match the connector Lambda's own 900s ceiling: giving up sooner abandons a scan
# that keeps running anyway. The heavy gold queries (GROUP BY over ~648K
# ENCOUNTERS, the PATIENTS join) are aggregations the connector cannot push
# down, so they stream every row and legitimately need minutes.
_GOLD_POLL_TIMEOUT_S = 900


def athena_rows(*, sql: str, catalog: str, database: str) -> tuple:
    """Run one query against the federated catalog and return (columns, rows).

    Paginates `get_query_results`: a single call caps at ~1000 rows, and a silently
    truncated gold result would make a correct agent answer look wrong. The header
    row Athena prepends to the FIRST page only is dropped.

    Parameters:
        sql: The SQL to execute.
        catalog: Athena catalog for QueryExecutionContext (the connector catalog).
        database: Athena database for QueryExecutionContext (the Snowflake schema).
    Returns:
        (columns, rows) where columns is list[str] and rows is
        list[list[str]] — every cell stringified, NULL rendered as ''. Matches the
        shape the query agent returns, so §7 can compare them directly.
    Raises:
        RuntimeError: The query failed, was cancelled, or did not finish in
            _GOLD_POLL_TIMEOUT_S seconds.
    """
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': database, 'Catalog': catalog},
        WorkGroup=ATHENA_WORKGROUP,
    )['QueryExecutionId']
    deadline = time.time() + _GOLD_POLL_TIMEOUT_S
    while True:
        ex = athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']
        state = ex['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        if time.time() > deadline:
            # Cancel before raising. Against a federated connector an abandoned query
            # keeps streaming rows for up to the connector Lambda's 900s ceiling, per
            # split, holding concurrency the whole time -- and the remote Snowflake
            # query outlives that too. Left running, one give-up here saturates the
            # connector and the NEXT query fails with a misleading TABLE_NOT_FOUND.
            try:
                athena.stop_query_execution(QueryExecutionId=qid)
                print(f"  ! cancelled gold query {qid} after {_GOLD_POLL_TIMEOUT_S}s")
            except Exception as cancel_err:
                print(f"  ! could NOT cancel gold query {qid}: {cancel_err}")
            raise RuntimeError(
                f"Gold query {qid} still {state} after {_GOLD_POLL_TIMEOUT_S}s "
                f"(cancelled): {sql}"
            )
        time.sleep(2)
    if state != 'SUCCEEDED':
        raise RuntimeError(
            f"Gold query {qid} {state}: "
            f"{ex['Status'].get('StateChangeReason', '(no reason given)')} | SQL: {sql}"
        )

    columns, rows, token, first_page = [], [], None, True
    while True:
        kwargs = {'QueryExecutionId': qid, 'MaxResults': 1000}
        if token:
            kwargs['NextToken'] = token
        page = athena.get_query_results(**kwargs)
        result_set = page['ResultSet']
        if first_page:
            columns = [c['Name'] for c in
                       result_set['ResultSetMetadata']['ColumnInfo']]
        page_rows = result_set['Rows']
        if first_page and page_rows:
            page_rows = page_rows[1:]      # drop Athena's header row (first page only)
        for r in page_rows:
            rows.append([d.get('VarCharValue', '') for d in r['Data']])
        first_page = False
        token = page.get('NextToken')
        if not token:
            break
    return columns, rows


def format_gold_answer(*, columns: list, rows: list) -> str:
    """Render a gold result set as the prose answer the SESSION judges compare against.

    A single scalar becomes just the number (so `FinalAnswerFaithfulness` compares
    an answer to a value, not to a JSON blob as in nb14); anything wider becomes
    'col=value, col=value' lines.

    Parameters:
        columns: Column names from `athena_rows`.
        rows: Row values from `athena_rows`.
    Returns:
        A compact human-readable rendering of the result set.
    """
    if not rows:
        return '(no rows)'
    if len(rows) == 1 and len(rows[0]) == 1:
        return rows[0][0]
    return '; '.join(
        ', '.join(f'{c}={v}' for c, v in zip(columns, row)) for row in rows
    )


for q in QUESTIONS:
    cols, grows = athena_rows(sql=q['gold_sql'], catalog=SNOWFLAKE_CATALOG,
                              database=SNOWFLAKE_SCHEMA)
    q['gold_columns'] = cols
    q['gold_rows'] = grows
    q['gold_answer'] = format_gold_answer(columns=cols, rows=grows)
    print(f"  {q['id']:<6} {len(grows)} row(s) → {q['gold_answer'][:110]}")

print(f"\n✓ Live gold ground truth computed for {len(QUESTIONS)} question(s)")

  sf-q1  1 row(s) → 10969


  sf-q2  2 row(s) → GENDER=female, patient_count=5539; GENDER=male, patient_count=5430


  sf-q3  1 row(s) → 100925


  sf-q4  5 row(s) → CONDITION_NAME=Medication review due (situation), condition_count=78310; CONDITION_NAME=Stress (finding), cond


  sf-q5  3 row(s) → CITY=Boston, patient_count=1023; CITY=Worcester, patient_count=292; CITY=Springfield, patient_count=270


  sf-q6  5 row(s) → ENCOUNTER_CLASS=AMB, encounter_count=605519; ENCOUNTER_CLASS=EMER, encounter_count=24082; ENCOUNTER_CLASS=IMP,


  sf-q7  2 row(s) → STATUS=completed, prescription_count=525241; STATUS=active, prescription_count=28181


  sf-q8  1 row(s) → 340847

✓ Live gold ground truth computed for 8 question(s)


## 4. Resolve the ontology_query runtime

In [9]:
# ── Look up the ontology_query runtime ARN ───────────────────────────────────
# The VKG query agent's output key is `QueryRuntimeArn` (NOT
# `OntologyQueryRuntimeArn`).
output_map = _get_stack_outputs('agentcore')
if not output_map:
    raise RuntimeError(f"AgentCore stack not found for project {PROJECT_NAME}")
ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ Query Runtime ARN : {_mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN)}")
print(f"✓ Agent ID          : {agent_id}")

✓ Query Runtime ARN : arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_ontology_query-CjvzJ6Ct1b
✓ Agent ID          : semantic_layer_dev_ontology_query-CjvzJ6Ct1b


## 5. Create the LLM-as-Judge evaluators (SESSION)

The same VKG judge family nb5 and nb14 use, so scores are comparable across
notebooks. `Builtin.GoalSuccessRate` is intentionally omitted for the reason nb5
documents: it mis-scores this deterministic-graph agent by mistaking an
intermediate intent/SPARQL span for the answer. The custom `GoalSuccess` judge
replaces it.

Unlike nb14, `FinalAnswerFaithfulness` **is** meaningful here — §3 gives it a prose
gold answer rather than a JSON gold-SQL blob.

In [10]:
# ── LLM-as-Judge (SESSION) evaluators — VKG family, from eval_judges ─────────
try:
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID

_cp = session.client('bedrock-agentcore-control', region_name=region)

# family='vkg' → [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded] (byte-identical
# to nb5's / nb14's judges).
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = create_custom_judges(
    control_client=_cp, family='vkg',
)
ALL_EVALUATOR_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print(f"✓ Custom SESSION LLM-as-Judge evaluators created (model {JUDGE_MODEL_ID}):")
print(f"  GoalSuccess             : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             : {SQL_GROUNDED_ID}")

# Display-name map for the results tables (event names drop the trailing -<hash>).
_EVAL_NAME = {
    GOAL_SUCCESS_ID: 'GoalSuccess', ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
    SQL_GROUNDED_ID: 'SqlGrounded',
}
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})

✓ Custom SESSION LLM-as-Judge evaluators created (model global.anthropic.claude-sonnet-5):
  GoalSuccess             : GoalSuccess_9c3e2e81-gkSvV0FUcJ
  FinalAnswerFaithfulness : FinalAnswerFaithfulness_9c3e2e81-xBwxUR4sTk
  SqlGrounded             : SqlGrounded_9c3e2e81-KNTffiBpZs


## 6. Build the Dataset + agent_invoker

One question = one `PredefinedScenario` = one session, all bound to the single
Snowflake layer via `ontologyId = LAYER_ID`. Each scenario's `Expected_Answer` is
the **prose gold** from §3, which `build_scenarios` threads onto both the final
`Turn.expected_response` and the SESSION assertions (via
`FINAL_ANSWER_ASSERTION_PREFIX`).

In [11]:
# Per-question chat-stream invoker + dataset builder (mirrors nb14, but binds ONE
# ontologyId and uses the live prose gold as the expected answer).
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

_specs = []
SCENARIO_Q = {}            # scenario_id -> the QUESTIONS entry (for the results join)
for i, q in enumerate(QUESTIONS):
    spec = parse_multiturn_row(
        {
            'Natural_Language_Question': q['question'],
            'Expected_Answer': q['gold_answer'],   # live prose gold from §3
            'Expected_Tier': 'vkg',                # these questions must route to VKG
        },
        index=i,
    )
    SCENARIO_Q[spec['scenario_id']] = q
    q['scenario_id'] = spec['scenario_id']
    _specs.append(spec)

scenarios = build_scenarios(_specs, ontology_id=LAYER_ID, simulated_enabled=False)
dataset = Dataset(scenarios=scenarios)
print(f"✓ Dataset: {len(dataset.scenarios)} scenario(s), all bound to {LAYER_ID}")


AGENT_RUNS = {}            # keyed by run_key(session_id, turn_idx)


def _scenario_id_for_session(sess: str) -> str:
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Invoke the VKG query agent for one Snowflake question via the chat-stream path.

    Binds ontologyId to the single Snowflake layer, parses the SSE body, and records
    the predicted SQL, returned rows, and cost/latency into AGENT_RUNS (§7 compares
    those rows against the live gold rows).

    Parameters:
        invoker_input: Framework-supplied payload (the question) + session id.
    Returns:
        AgentInvokerOutput carrying the judge-visible answer text.
    """
    sid = invoker_input.session_id
    scenario_id = _scenario_id_for_session(sid)
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=LAYER_ID, turn_idx=turn_idx,
                                 user_id="snowflake-eval", source="eval")
    payload["mode"] = "vkg"                    # VKG path (default is semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001 — one question failing must not abort the batch
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "scenario_id": scenario_id, "turn_idx": turn_idx,
        "ontology_id": LAYER_ID, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "executed_sql": parsed.get("executed_sql", ""), "degraded": parsed.get("degraded"),
        "rows": parsed.get("rows", []) or [], "row_count": parsed.get("row_count"),
        "provenance": parsed.get("provenance"), "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "answer" if parsed.get("sql") else ("clarify" if parsed.get("clarification") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{scenario_id}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))

✓ Dataset: 8 scenario(s), all bound to vkg-snowflake_fhir_r4


## 7. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent once per question (via `agent_invoker`),
waits `ingestion_delay_seconds` for CloudWatch span ingestion, submits ONE
`StartBatchEvaluation` over the three SESSION judges, and polls to completion.

In [12]:
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")
print(f"Evaluators   : {ALL_EVALUATOR_IDS}")
print(f"Scenarios    : {len(dataset.scenarios)}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=420,   # give spans (incl. the SQL-exec span) time to land
)

batch_runner = BatchEvaluationRunner(region=region)

cfg = BatchEvaluationRunConfig(
    batch_evaluation_name=f"snowflake_vkg_qa_{uuid.uuid4().hex[:8]}",
    description="Snowflake federated VKG query-agent QA: SESSION LLM judges "
                "(GoalSuccess / FinalAnswerFaithfulness / SqlGrounded).",
    evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
    data_source=batch_data_source,
    max_concurrent_scenarios=3,    # synchronous agent, up to 900s/question
    polling_timeout_seconds=5400,
    polling_interval_seconds=30,
)

print(f"\n=== {cfg.batch_evaluation_name} ===")
print("Invoking the agent per question, then evaluating server-side...")
batch_result = batch_runner.run_dataset_evaluation(
    config=cfg, dataset=dataset, agent_invoker=agent_invoker)
print(f"\n✓ Batch status: {batch_result.status}")
if getattr(batch_result, 'agent_invocation_failures', None):
    print(f"⚠ Agent invocation failures: {len(batch_result.agent_invocation_failures)}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")

SERVICE_NAME : semantic_layer_dev_ontology_query-CjvzJ6Ct1b.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology_query-CjvzJ6Ct1b-DEFAULT
Evaluators   : ['GoalSuccess_9c3e2e81-gkSvV0FUcJ', 'FinalAnswerFaithfulness_9c3e2e81-xBwxUR4sTk', 'SqlGrounded_9c3e2e81-KNTffiBpZs']
Scenarios    : 8



=== snowflake_vkg_qa_609ad62b ===
Invoking the agent per question, then evaluating server-side...


  OK [gt-row-00] 38.25s . answer


  OK [gt-row-02] 39.16s . answer


  OK [gt-row-01] 50.61s . answer


  OK [gt-row-04] 33.61s . answer


  OK [gt-row-03] 38.54s . answer


  OK [gt-row-05] 37.49s . answer


  OK [gt-row-06] 35.15s . answer


  OK [gt-row-07] 58.3s . answer



✓ Batch status: COMPLETED


## 8. Local Execution Accuracy — agent rows vs. live gold rows

This is the objective signal, replacing nb14's `codeBased` exec-metric evaluators
(see the intro for why they can't run against a federated catalog).

**Comparison rule.** Each result set is reduced to a multiset of rows, and each row
to a **sorted** tuple of normalized cells. That makes the comparison insensitive to
row order and to column order/aliasing (the agent may legitimately return
`GENDER, n` where the gold returns `n, GENDER`) while remaining sensitive to the
actual values — which is what execution accuracy is meant to measure. Numeric-looking
cells are compared as floats rounded to 4 dp so `5539` and `5539.0` match.

A question whose agent run errored, was degraded, or returned no rows scores 0 and
is reported as such — never silently skipped.

In [13]:
def _norm_cell(value) -> str:
    """Normalize one result cell for comparison.

    Numeric-looking cells are compared as floats rounded to 4 dp (so '5539' matches
    '5539.0' and '69.38' matches '69.380'); everything else is compared as a
    stripped string. NULL/None becomes ''.

    Parameters:
        value: A single cell from either result set (any scalar type).
    Returns:
        The normalized cell as a string.
    """
    if value is None:
        return ''
    text = str(value).strip()
    try:
        number = float(text)
    except ValueError:
        return text
    # 'nan'/'inf' parse as floats but nan != nan would make a row never match
    # itself, so leave non-finite values as text and compare them literally.
    if number != number or number in (float('inf'), float('-inf')):
        return text
    return f'{round(number, 4):.4f}'


def _norm_rows(rows) -> list:
    """Reduce a result set to a sorted multiset of order-insensitive normalized rows.

    Each row becomes a SORTED tuple of normalized cells, so column order/aliasing
    differences between the agent's SQL and the gold SQL do not count as errors;
    the outer list is sorted so row order does not either.

    Parameters:
        rows: list of rows; each row a list/tuple of cells, or a dict (values used),
            or a bare scalar (treated as a 1-cell row).
    Returns:
        A sorted list of tuples suitable for equality comparison.
    """
    out = []
    for row in rows or []:
        if isinstance(row, dict):
            cells = list(row.values())
        elif isinstance(row, (list, tuple)):
            cells = list(row)
        else:
            cells = [row]
        out.append(tuple(sorted(_norm_cell(c) for c in cells)))
    return sorted(out)


# Map each scenario's agent run back to its question and score it.
_runs_by_scenario = {rec['scenario_id']: rec for rec in AGENT_RUNS.values()}

ex_rows = []
for q in QUESTIONS:
    rec = _runs_by_scenario.get(q.get('scenario_id'))
    gold_norm = _norm_rows(q['gold_rows'])
    if rec is None:
        ex_rows.append({'id': q['id'], 'difficulty': q['difficulty'],
                        'question': q['question'], 'match': 0,
                        'reason': 'no agent run recorded for this scenario',
                        'agent_rows': None, 'gold_rows': len(q['gold_rows']),
                        'agent_sql': '', 'gold_answer': q['gold_answer'],
                        'agent_answer': ''})
        continue
    agent_norm = _norm_rows(rec['rows'])
    if rec['invoke_error']:
        match, reason = 0, f"invoke error: {rec['invoke_error']}"
    elif rec['degraded']:
        match, reason = 0, f"degraded: {rec['degraded']}"
    elif not agent_norm:
        match, reason = 0, 'agent returned no rows'
    elif agent_norm == gold_norm:
        match, reason = 1, 'exact result-set match'
    else:
        match, reason = 0, 'result set differs from gold'
    ex_rows.append({
        'id': q['id'], 'difficulty': q['difficulty'], 'question': q['question'],
        'match': match, 'reason': reason,
        'agent_rows': len(agent_norm), 'gold_rows': len(gold_norm),
        'agent_sql': rec['agent_sql'], 'gold_answer': q['gold_answer'],
        'agent_answer': (rec['answer'] or '')[:400],
    })

df_ex = pd.DataFrame(ex_rows)
_n = len(ex_rows)
LOCAL_EX = round(sum(r['match'] for r in ex_rows) / _n, 4) if _n else None
ex_by_difficulty = {}
for r in ex_rows:
    ex_by_difficulty.setdefault(r['difficulty'], []).append(r['match'])
ex_by_difficulty = {k: round(sum(v) / len(v), 4) for k, v in ex_by_difficulty.items()}

print(f"Local ExecutionAccuracy: {LOCAL_EX}  ({sum(r['match'] for r in ex_rows)}/{_n})")
for band in sorted(ex_by_difficulty):
    print(f"  {band:>12}: {ex_by_difficulty[band]}")
display(df_ex[['id', 'difficulty', 'match', 'reason', 'agent_rows', 'gold_rows']])

Local ExecutionAccuracy: 1.0  (8/8)
   challenging: 1.0
      moderate: 1.0
        simple: 1.0


,id,difficulty,match,reason,agent_rows,gold_rows
0,sf-q1,simple,1,exact result-set match,1,1
1,sf-q2,simple,1,exact result-set match,2,2
2,sf-q3,simple,1,exact result-set match,1,1
3,sf-q4,moderate,1,exact result-set match,5,5
4,sf-q5,moderate,1,exact result-set match,3,3
5,sf-q6,moderate,1,exact result-set match,5,5
6,sf-q7,moderate,1,exact result-set match,2,2
7,sf-q8,challenging,1,exact result-set match,1,1


## 9. Results — judge aggregates + per-question detail, persisted

In [14]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores (LLM judges) ───────────────────────────
agg_rows, _agg_by_name = [], {}
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (stats.average_score if stats and stats.average_score is not None else None)
        name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
        _agg_by_name[name] = round(avg, 4) if avg is not None else None
        agg_rows.append({
            'Evaluator': name,
            'AvgScore': f"{avg:.3f}" if avg is not None else None,
            'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

# ── 2. Per-question evaluator events ─────────────────────────────────────────
def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-question events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    sid = _scenario_id_for_session(sess)
    _q = SCENARIO_Q.get(sid, {})
    event_rows.append({
        'scenario_id': sid,
        'id': _q.get('id', ''),
        'difficulty': _q.get('difficulty', ''),
        'question': _q.get('question', ''),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': str(_event_field(e, 'gen_ai.evaluation.explanation') or ''),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-question evaluator events: {len(df_events)}")

# ── 3. Per-question agent detail (predicted SQL / provenance / latency) ──────
agent_detail = [{
    'scenario_id': rec['scenario_id'], 'question': rec['message'],
    'agent_sql': rec['agent_sql'], 'executed_sql': rec['executed_sql'],
    'provenance': rec['provenance'], 'degraded': rec['degraded'],
    'row_count': rec['row_count'], 'wall_clock_s': rec['wall_clock_s'],
    'error': rec['invoke_error'],
} for rec in AGENT_RUNS.values()]
df_agent = pd.DataFrame(agent_detail)

# ── Persist ──────────────────────────────────────────────────────────────────
combined_file = f"../data/eval/results/snowflake_vkg_query_eval_{timestamp}.json"
combined = {
    'source': 'snowflake_federated',
    'catalog': SNOWFLAKE_CATALOG,
    'schema': SNOWFLAKE_SCHEMA,
    'layer_id': LAYER_ID,
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': _mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN),
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'llm_judges': {'GoalSuccess': GOAL_SUCCESS_ID,
                   'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                   'SqlGrounded': SQL_GROUNDED_ID},
    'llm_judge_aggregates': {k: _agg_by_name.get(k) for k in
                             ('GoalSuccess', 'FinalAnswerFaithfulness', 'SqlGrounded')},
    # Local exec accuracy replaces the codeBased evaluators, which cannot execute
    # gold SQL against a federated catalog (see the notebook intro).
    'local_execution_accuracy': LOCAL_EX,
    'local_execution_accuracy_by_difficulty': ex_by_difficulty,
    'exec_metrics_note': (
        'EX computed in-notebook by executing each gold query on Athena and '
        'comparing result sets (order-insensitive within and across rows). The '
        'BenchmarkEvalHandler codeBased evaluators are NOT used: their role cannot '
        'invoke the Snowflake connector Lambda.'
    ),
    'aggregate_summaries': agg_rows,
    'per_question_events': event_rows,
    'per_question_execution': ex_rows,
    'agent_detail': agent_detail,
    'questions': [{k: v for k, v in q.items() if k != 'gold_rows'} for q in QUESTIONS],
    'n_questions': len(QUESTIONS),
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores (SESSION judges) ===")
display(df_agg)
print(f"\nHeadline (n={len(QUESTIONS)}): local EX={LOCAL_EX}  "
      f"GoalSuccess={_agg_by_name.get('GoalSuccess')}  "
      f"FinalAnswerFaithfulness={_agg_by_name.get('FinalAnswerFaithfulness')}  "
      f"SqlGrounded={_agg_by_name.get('SqlGrounded')}")
print("\n=== Per-question evaluator scores ===")
display(df_events)
print("=== Per-question agent detail ===")
display(df_agent)

Batch ID : snowflake_vkg_qa_609ad62b-fcd3578d05
Status   : COMPLETED

Sessions: completed=8 failed=0 total=8



Per-question evaluator events: 24

✓ Wrote ../data/eval/results/snowflake_vkg_query_eval_20260813_143254.json

=== Aggregate per-evaluator scores (SESSION judges) ===


,Evaluator,AvgScore,Evaluated,Failed
0,SqlGrounded,1.000,8,0
1,GoalSuccess,1.000,8,0
2,FinalAnswerFaithfulness,1.000,8,0



Headline (n=8): local EX=1.0  GoalSuccess=1.0  FinalAnswerFaithfulness=1.0  SqlGrounded=1.0

=== Per-question evaluator scores ===


,scenario_id,id,difficulty,question,evaluator,score,label,explanation
0,gt-row-04,sf-q5,moderate,Which three cities have the most patients?,SqlGrounded,1.0,pass,"No executed SQL is present in the context; only a SliceSufficiency tool call and the final answer are shown. Per instructions, when there is no executed SQL surfaced in the context, the agent is considered to have degraded rather than run ungrounded SQL, so the grounding invariant is upheld."
1,gt-row-04,sf-q5,moderate,Which three cities have the most patients?,GoalSuccess,1.0,pass,"The agent's final answer states Boston (1,023), Worcester (292), and Springfield (270) as the three cities with the most patients, matching exactly the expected assertion values for CITY and patient_count. This is a substantive, factually correct answer with no clarification needed."
2,gt-row-04,sf-q5,moderate,Which three cities have the most patients?,FinalAnswerFaithfulness,1.0,pass,"The final answer states Boston (1023), Worcester (292), Springfield (270), which exactly matches the expected answer's cities and counts."
3,gt-row-00,sf-q1,simple,How many patients are in the dataset?,GoalSuccess,1.0,pass,"The agent's final response states 'There are 10,969 patients in the dataset.' This matches the expected answer of 10969 exactly, derived from the query result row [""10969""]. The answer is substantive and consistent with the assertion."
4,gt-row-00,sf-q1,simple,How many patients are in the dataset?,FinalAnswerFaithfulness,1.0,pass,"The agent's final answer states there are 10,969 patients in the dataset, which matches the expected answer of 10969 exactly."
5,gt-row-00,sf-q1,simple,How many patients are in the dataset?,SqlGrounded,1.0,pass,"No executed SQL is present in the context; only the retrieved ontology context and the final result (10969 patients) are shown, with no Phase 5 SQL surfaced. Per the instructions, absence of executed SQL indicates a degraded run where the agent could not ground SQL, which upholds the grounding invariant by default. However, the result strongly implies a COUNT(*) FROM PATIENTS query was run, and PATIENTS/id maps cleanly to the ontology's Patient class. Since no SQL is explicitly shown to inspect, treat this as the 'no SQL executed (degraded)' case per instructions, which passes."
6,gt-row-03,sf-q4,moderate,"What are the five most common condition names, and how many times does each occur?",FinalAnswerFaithfulness,1.0,pass,"The agent's final answer lists exactly the five condition names and counts specified in the expected answer: Medication review due (78310), Stress (30445), Gingivitis (28972), Full-time employment (28231), Part-time employment (17762). All values match the ground truth exactly."
7,gt-row-03,sf-q4,moderate,"What are the five most common condition names, and how many times does each occur?",GoalSuccess,1.0,pass,"The agent's final response reports the top 5 condition names with exact counts matching the expected assertion: Medication review due (situation)=78310, Stress (finding)=30445, Gingivitis (disorder)=28972, Full-time employment (finding)=28231, Part-time employment (finding)=17762. This matches the expected answer exactly, both in entities and counts. No clarification was needed and none was requested inappropriately. The response is substantive and correctly derived from the executed query results."
8,gt-row-03,sf-q4,moderate,"What are the five most common condition names, and how many times does each occur?",SqlGrounded,1.0,pass,"No executed SQL is present in the context; only the ontology schema and the tool call SliceSufficiency (which just validates the ontology slice) are shown, along with the final result table. Since there is no Phase 5 SQL surfaced in the conversation, the agent's grounding cannot be checked against a query, meaning either the SQL was not exposed or the run was degraded. Per instructions, absence of executed SQL means the grounding invariant is upheld by default, scoring as pass."
9,gt-row-05,sf-q6,moder

=== Per-question agent detail ===


,scenario_id,question,agent_sql,executed_sql,provenance,degraded,row_count,wall_clock_s,error
0,gt-row-00,How many patients are in the dataset?,SELECT (COUNT(?p) AS ?n)\nWHERE {\n ?p a <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Patient> .\n},"SELECT COUNT(*) AS ""v0""\nFROM (SELECT DISTINCT R1.ID AS ""ID1m52""\nFROM ""FHIR_R4"".""PATIENTS"" R1\nWHERE R1.ID IS NOT NULL\n) R3\n","{'tier': 'vkg', 'grounded': True, 'sources': ['class:Condition', 'class:Encounter', 'class:FhirResource', 'class:Medication', 'class:Observation', 'class:Patient', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:partition', 'class:patient_ref', 'class:snomed_code', 'class:patient_ref', 'class:id', 'class:patient_ref', 'class:patient_ref', 'class:resource'], 'degraded': None}",None,1,38.25,None
1,gt-row-02,How many conditions have a clinical status of active?,"SELECT (COUNT(*) AS ?n)\nWHERE {\n ?c a <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Condition> ;\n <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Condition/clinical_status> ""active"" .\n}","SELECT COUNT(*) AS ""v0""\nFROM (SELECT DISTINCT R1.ID AS ""ID1m1""\nFROM ""FHIR_R4"".""CONDITIONS"" R1\nWHERE (R1.ID IS NOT NULL AND 'active' = R1.CLINICAL_STATUS)\n) R3\n","{'tier': 'vkg', 'grounded': True, 'sources': ['class:Condition', 'class:Encounter', 'class:FhirResource', 'class:Observation', 'class:Patient', 'class:Medication', 'class:clinical_status', 'class:status', 'class:abatement_date', 'class:resource', 'class:resource', 'class:last_updated', 'class:partition', 'class:patient_ref', 'class:hasPatients', 'class:id', 'class:last_updated', 'class:partition'], 'degraded': None}",None,1,39.16,None
2,gt-row-01,How many patients are there of each gender?,SELECT ?gender (COUNT(?p) AS ?n)\nWHERE {\n ?p a <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Patient> ;\n <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Patient/gender> ?gender .\n}\nGROUP BY ?gender,"SELECT R3.""GENDER1m56"" AS ""GENDER1m56"", COUNT(*) AS ""v1""\nFROM (SELECT DISTINCT R1.GENDER AS ""GENDER1m56"", R1.ID AS ""ID1m52""\nFROM ""FHIR_R4"".""PATIENTS"" R1\nWHERE (R1.GENDER IS NOT NULL AND R1.ID IS NOT NULL)\n) R3\nGROUP BY R3.""GENDER1m56""\n","{'tier': 'vkg', 'grounded': True, 'sources': ['class:Condition', 'class:Encounter', 'class:FhirResource', 'class:Medication', 'class:Observation', 'class:Patient', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:gender', 'class:patient_ref', 'class:snomed_code', 'class:patient_ref', 'class:id', 'class:patient_ref', 'class:patient_ref', 'class:state'], 'degraded': None}",None,2,50.61,None
3,gt-row-04,Which three cities have the most patients?,SELECT ?city (COUNT(?p) AS ?patientCount) WHERE {\n ?p a <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Patient> ;\n <http://snowflake-fhir_r4-ontology-20260813-030435/ontology/vkg-snowflake_fhir_r4/Patient/city> ?city .\n}\nGROUP BY ?city\nORDER BY DESC(?patientCount)\nLIMIT 3,"SELECT R3.""CITY1m54"" AS ""CITY1m54"", COUNT(*) AS ""v1""\nFROM (SELECT DISTINCT R1.CITY AS ""CITY1m54"", R1.ID AS ""ID1m52""\nFROM ""FHIR_R4"".""PATIENTS"" R1\nWHERE (R1.CITY IS NOT NULL AND R1.ID IS NOT NULL)\n) R3\nGROUP BY R3.""CITY1m54""\nORDER BY COUNT(*) DESC NULLS LAST\nLIMIT 3","{'tier': 'vkg', 'grounded': True, 'sources': ['class:Condition', 'class:Encounter', 'class:FhirResource', 'class:Medication', 'class:Observation', 'class:Patient', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:hasPatients', 'class:patient_ref', 'class:snomed_code', 'class:patient_ref', 'class:id', 'class:patient_ref', 'class:patient_ref', 'class:city', 'class:state'], 'degraded': None}",None,3,33.61,None
4,gt-row-03,"What are the five most common condition names, and how many times 

## Summary

This notebook runs a hand-authored QA set through the deployed **ontology_query**
(VKG) agent against the **Snowflake federated** semantic layer built by
`17_snowflake_semantic_layer.ipynb`, and scores it two ways:

1. **LLM judges (SESSION, server-side)** — `GoalSuccess`,
   `FinalAnswerFaithfulness`, `SqlGrounded` from `eval_judges` (VKG family), the
   same judges nb5 and nb14 use, so scores are comparable across notebooks.
2. **Local ExecutionAccuracy** — each gold query is executed on Athena from this
   notebook and its result set compared to the agent's returned rows
   (order-insensitive within and across rows, numeric cells compared as floats).

### Layer binding
`ontologyId = LAYER_ID = f"vkg-snowflake_{SNOWFLAKE_ATHENA_SCHEMA.lower()}"` →
`vkg-snowflake_fhir_r4`. Derived from the same two constants nb17 used, then
verified `completed` in DynamoDB before any question is asked.

### Reading the results
| Pattern | Likely meaning |
|:--|:--|
| High judge scores, low local EX | the agent produced fluent answers over wrong numbers — trust EX |
| `degraded` set on most rows | the SQL gate rejected the generated query; read `agent_sql` vs `executed_sql` |
| `sf-q8` fails while q1–q7 pass | the prefixed foreign key (`'urn:uuid:' || ID`) is not expressed in the generated OBDA mapping — an ontology-modelling gap, not a connector problem |
| Everything returns 0 rows | check the connector: a redeploy resets the connector Lambda's environment (`npm run set:snowflake-casing`) |

### Known gaps
- The three `codeBased` exec-metric evaluators (EX / R-VES / Soft-F1) used for the
  S3-Tables benchmarks are **not** registered here. The `BenchmarkEvalHandler*`
  roles carry the full Athena + Glue + `s3tables` set but no `lambda:` action, and
  Athena invokes a federated connector as the query's own principal — so they
  cannot execute gold SQL against `snowflake_*`. Adding `lambda:InvokeFunction` on
  the connector function to those roles would let nb14's exact evaluator setup be
  reused, and would also restore R-VES and Soft-F1, which this notebook does not
  compute.
- The question set is small and hand-authored; it is a live QA gate on the
  federated path, not a benchmark. Treat the absolute EX as a smoke-test signal.

### Output
`../data/eval/results/snowflake_vkg_query_eval_*.json` — judge aggregates, local EX
(overall + per difficulty), per-question evaluator events, and per-question agent
detail (predicted SQL, provenance, latency).
